In [1]:
!pip install fastapi uvicorn python-multipart -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.6.0 requires typing-extensions~=3.7.4, but you have typing-extensions 4.13.2 which is incompatible.


In [1]:
import fastapi
import uvicorn
print("FastAPI version:", fastapi.__version__)
print("Uvicorn version:", uvicorn.__version__)

FastAPI version: 0.124.4
Uvicorn version: 0.33.0


In [2]:
from pathlib import Path

print("best_model.h5    :", Path("saved_model/best_model.h5").exists())
print("class_indices.json:", Path("saved_model/class_indices.json").exists())
print("model_info.json  :", Path("saved_model/model_info.json").exists())

best_model.h5    : True
class_indices.json: True
model_info.json  : True


In [ ]:
# ─────────────────────────────────────────────
# AI Document Classifier — FastAPI Backend
# ─────────────────────────────────────────────

import uvicorn
import numpy as np
import json
import io
import os
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

CONFIG = {
    "model_path":       "saved_model/best_model.h5",
    "class_indices":    "saved_model/class_indices.json",
    "model_info":       "saved_model/model_info.json",
    "image_size":       (224, 224),
    "allowed_types":    ["image/jpeg", "image/png", "image/jpg", "image/bmp", "image/webp"],
    "host":             "127.0.0.1",
    "port":             8000,
}

# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────

print("Loading model...")
model = load_model(CONFIG["model_path"])

with open(CONFIG["class_indices"], "r") as f:
    class_indices = json.load(f)

with open(CONFIG["model_info"], "r") as f:
    model_info = json.load(f)

class_names = {v: k for k, v in class_indices.items()}
print(f"✔ Model loaded! Classes: {class_names}")

# ─────────────────────────────────────────────
# FASTAPI APP
# ─────────────────────────────────────────────

app = FastAPI(
    title="AI Document Classifier API",
    description="""
## MPOnline AIML Internship Project

This API classifies uploaded document images into:
- **Aadhar Card**
- **PAN Card**
- **Marksheet**

### Features:
- Upload any document image
- Get predicted class with confidence score
- View all class probabilities
- Health check endpoint
- Model information endpoint
    """,
    version="1.0.0",
    contact={
        "name": "MPOnline AIML Intern",
    },
)

# Allow all origins for development
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────────
# HELPER FUNCTION
# ─────────────────────────────────────────────

def preprocess_image(img_bytes: bytes):
    """Convert raw image bytes to model-ready tensor."""
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img = img.resize(CONFIG["image_size"])
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def run_prediction(img_array):
    """Run model prediction and return results."""
    predictions = model.predict(img_array, verbose=0)
    predicted_index = int(np.argmax(predictions[0]))
    predicted_class = class_names[predicted_index]
    confidence = float(predictions[0][predicted_index]) * 100

    all_probs = {
        class_names[i]: round(float(predictions[0][i]) * 100, 2)
        for i in range(len(class_names))
    }

    return predicted_class, confidence, all_probs


# ─────────────────────────────────────────────
# API ROUTES
# ─────────────────────────────────────────────

# 1. ROOT
@app.get("/", tags=["General"])
def root():
    return {
        "message":  "Welcome to AI Document Classifier API!",
        "project":  "MPOnline AIML Internship",
        "version":  "1.0.0",
        "docs":     "http://127.0.0.1:8000/docs",
        "endpoints": {
            "health":     "GET  /health",
            "model_info": "GET  /model-info",
            "upload":     "POST /upload",
            "predict":    "POST /predict",
        }
    }


# 2. HEALTH CHECK
@app.get("/health", tags=["Health"])
def health_check():
    """Check if the API is running properly."""
    return {
        "status":    "healthy",
        "timestamp": datetime.now().isoformat(),
        "model":     "loaded",
        "framework": f"TensorFlow {tf.__version__}",
    }


# 3. MODEL INFO
@app.get("/model-info", tags=["Model"])
def get_model_info():
    """Get details about the trained model."""
    return {
        "model_name":     model_info.get("model_name", "MobileNetV2"),
        "framework":      model_info.get("framework", "TensorFlow/Keras"),
        "val_accuracy":   f"{model_info.get('val_accuracy', 0)}%",
        "val_loss":       model_info.get("val_loss", 0),
        "total_params":   model_info.get("total_params", 0),
        "epochs_trained": model_info.get("epochs_trained", 0),
        "input_size":     "224x224",
        "classes":        list(class_indices.keys()),
        "num_classes":    len(class_indices),
    }


# 4. UPLOAD DOCUMENT
@app.post("/upload", tags=["Document"])
async def upload_document(file: UploadFile = File(...)):
    """
    Upload a document image.
    - Accepts: JPG, PNG, BMP, WEBP
    - Returns: File info and upload confirmation
    """
    # Validate file type
    if file.content_type not in CONFIG["allowed_types"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type '{file.content_type}'. Allowed: JPG, PNG, BMP, WEBP"
        )

    # Read file
    contents = await file.read()
    file_size = len(contents) / 1024  # KB

    return {
        "status":      "success",
        "message":     "Document uploaded successfully!",
        "filename":    file.filename,
        "file_type":   file.content_type,
        "file_size":   f"{file_size:.2f} KB",
        "timestamp":   datetime.now().isoformat(),
        "next_step":   "Send this file to POST /predict to classify it",
    }


# 5. PREDICT DOCUMENT
@app.post("/predict", tags=["Document"])
async def predict_document(file: UploadFile = File(...)):
    """
    Upload and classify a document image.
    - Accepts: JPG, PNG, BMP, WEBP
    - Returns: Predicted class + confidence score + all probabilities
    """
    # Validate file type
    if file.content_type not in CONFIG["allowed_types"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type '{file.content_type}'. Allowed: JPG, PNG, BMP, WEBP"
        )

    try:
        # Read and preprocess image
        contents = await file.read()
        img_array = preprocess_image(contents)

        # Run prediction
        predicted_class, confidence, all_probs = run_prediction(img_array)

        # Confidence level label
        if confidence >= 90:
            confidence_level = "Very High"
        elif confidence >= 75:
            confidence_level = "High"
        elif confidence >= 60:
            confidence_level = "Medium"
        else:
            confidence_level = "Low"

        return {
            "status":           "success",
            "filename":         file.filename,
            "predicted_class":  predicted_class,
            "confidence":       round(confidence, 2),
            "confidence_level": confidence_level,
            "all_probabilities": all_probs,
            "timestamp":        datetime.now().isoformat(),
            "model_used":       model_info.get("model_name", "MobileNetV2"),
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed: {str(e)}"
        )


# ─────────────────────────────────────────────
# RUN SERVER
# ─────────────────────────────────────────────

print("\n" + "═" * 50)
print("  🚀 Starting FastAPI Server...")
print("  📌 URL        : http://127.0.0.1:8000")
print("  📄 Swagger UI : http://127.0.0.1:8000/docs")
print("  📘 Redoc      : http://127.0.0.1:8000/redoc")
print("═" * 50 + "\n")

config = uvicorn.Config(
    app,
    host=CONFIG["host"],
    port=CONFIG["port"],
    log_level="info"
)
server = uvicorn.Server(config)
await server.serve()

Loading model...
✔ Model loaded! Classes: {0: 'Aadhar_Card', 1: 'Marksheet', 2: 'Pan_Card'}

══════════════════════════════════════════════════
  🚀 Starting FastAPI Server...
  📌 URL        : http://127.0.0.1:8000
  📄 Swagger UI : http://127.0.0.1:8000/docs
  📘 Redoc      : http://127.0.0.1:8000/redoc
══════════════════════════════════════════════════



INFO:     Started server process [80722]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:57634 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:57634 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:57657 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:57661 - "GET /model-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:57678 - "POST /predict HTTP/1.1" 200 OK


In [3]:
import uvicorn
import numpy as np
import json
import io
import os
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import load_model

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

CONFIG = {
    "models": {
        "MobileNetV2": {
            "model_path":    "saved_model/best_model.h5",
            "class_indices": "saved_model/class_indices.json",
            "model_info":    "saved_model/model_info.json",
        },
        "ResNet50": {
            "model_path":    "saved_model_resnet/best_model_resnet.h5",
            "class_indices": "saved_model_resnet/class_indices.json",
            "model_info":    "saved_model_resnet/model_info.json",
        },
        "VGG16": {
            "model_path":    "saved_model_vgg16/best_model_vgg16.h5",
            "class_indices": "saved_model_vgg16/class_indices.json",
            "model_info":    "saved_model_vgg16/model_info.json",
        },
    },
    "default_model": "MobileNetV2",
    "image_size":    (224, 224),
    "allowed_types": ["image/jpeg", "image/png", "image/jpg", "image/bmp", "image/webp"],
    "host":          "127.0.0.1",
    "port":          8001,  # new port to avoid conflict with old server
}

# ─────────────────────────────────────────────
# MODEL MANAGER
# ─────────────────────────────────────────────

class ModelManager:
    def __init__(self):
        self.current_model_name = CONFIG["default_model"]
        self.model              = None
        self.class_names        = None
        self.model_info         = None
        self.loaded_models      = {}   # cache loaded models
        self._load_model(self.current_model_name)

    def _load_model(self, model_name: str):
        """Load a model by name. Uses cache if already loaded."""
        if model_name in self.loaded_models:
            # Use cached model
            self.model, self.class_names, self.model_info = self.loaded_models[model_name]
            self.current_model_name = model_name
            print(f"✔ Switched to cached model: {model_name}")
            return

        cfg = CONFIG["models"][model_name]

        # Check files exist
        for key, path in cfg.items():
            if not Path(path).exists():
                raise FileNotFoundError(f"File not found: {path}")

        print(f"Loading {model_name}...")
        model = load_model(cfg["model_path"])

        with open(cfg["class_indices"], "r") as f:
            class_indices = json.load(f)
        class_names = {v: k for k, v in class_indices.items()}

        with open(cfg["model_info"], "r") as f:
            model_info = json.load(f)

        # Cache it
        self.loaded_models[model_name] = (model, class_names, model_info)

        self.model              = model
        self.class_names        = class_names
        self.model_info         = model_info
        self.current_model_name = model_name

        print(f"✔ {model_name} loaded! Accuracy: {model_info.get('val_accuracy')}%")

    def switch_model(self, model_name: str):
        """Switch to a different model."""
        if model_name not in CONFIG["models"]:
            raise ValueError(f"Model '{model_name}' not found. Available: {list(CONFIG['models'].keys())}")
        self._load_model(model_name)

    def predict(self, img_array):
        """Run prediction using current model."""
        predictions = self.model.predict(img_array, verbose=0)
        predicted_index = int(np.argmax(predictions[0]))
        predicted_class = self.class_names[predicted_index]
        confidence      = float(predictions[0][predicted_index]) * 100
        all_probs       = {
            self.class_names[i]: round(float(predictions[0][i]) * 100, 2)
            for i in range(len(self.class_names))
        }
        return predicted_class, confidence, all_probs


# Initialize model manager
print("═" * 50)
print("  AI Document Classifier — FastAPI Backend")
print("  With Model Switching Support")
print("═" * 50 + "\n")

manager = ModelManager()

# ─────────────────────────────────────────────
# FASTAPI APP
# ─────────────────────────────────────────────

app = FastAPI(
    title="AI Document Classifier API",
    description="""
## MPOnline AIML Internship Project

This API classifies uploaded document images into:
- **Aadhar Card**
- **PAN Card**
- **Marksheet**

### Supported Models:
| Model | Accuracy | Params |
|---|---|---|
| MobileNetV2 | 97.33% | 2.4M |
| ResNet50 | 72.33% | 24M |
| VGG16 | - | 15M |

### Features:
- Upload any document image
- Get predicted class with confidence score
- **Switch between MobileNetV2, ResNet50, VGG16**
- Health check endpoint
- Model information endpoint
    """,
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────

def preprocess_image(img_bytes: bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img = img.resize(CONFIG["image_size"])
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def get_confidence_level(confidence: float) -> str:
    if confidence >= 90:   return "Very High"
    elif confidence >= 75: return "High"
    elif confidence >= 60: return "Medium"
    else:                  return "Low"


# ─────────────────────────────────────────────
# API ROUTES
# ─────────────────────────────────────────────

# 1. ROOT
@app.get("/", tags=["General"])
def root():
    return {
        "message":       "Welcome to AI Document Classifier API!",
        "version":       "2.0.0",
        "current_model": manager.current_model_name,
        "available_models": list(CONFIG["models"].keys()),
        "docs":          "http://127.0.0.1:8001/docs",
        "endpoints": {
            "health":       "GET  /health",
            "model_info":   "GET  /model-info",
            "switch_model": "POST /switch-model/{model_name}",
            "upload":       "POST /upload",
            "predict":      "POST /predict",
            "compare":      "POST /compare",
        }
    }


# 2. HEALTH CHECK
@app.get("/health", tags=["Health"])
def health_check():
    """Check if the API and model are running properly."""
    return {
        "status":          "healthy",
        "timestamp":       datetime.now().isoformat(),
        "model_loaded":    manager.current_model_name,
        "available_models": list(CONFIG["models"].keys()),
        "framework":       f"TensorFlow {tf.__version__}",
    }


# 3. MODEL INFO
@app.get("/model-info", tags=["Model"])
def get_model_info():
    """Get details about the currently active model."""
    info = manager.model_info
    return {
        "current_model":    manager.current_model_name,
        "available_models": list(CONFIG["models"].keys()),
        "model_name":       info.get("model_name", manager.current_model_name),
        "framework":        info.get("framework", "TensorFlow/Keras"),
        "val_accuracy":     f"{info.get('val_accuracy', 0)}%",
        "val_loss":         info.get("val_loss", 0),
        "total_params":     info.get("total_params", 0),
        "epochs_trained":   info.get("epochs_trained", 0),
        "input_size":       "224x224",
        "classes":          list(manager.class_names.values()),
        "num_classes":      len(manager.class_names),
    }


# 4. SWITCH MODEL ← New!
@app.post("/switch-model/{model_name}", tags=["Model"])
def switch_model(model_name: str):
    """
    Switch between available models.
    - **MobileNetV2** — 97.33% accuracy (recommended)
    - **ResNet50**    — 72.33% accuracy
    - **VGG16**       — run VGG16 training first
    """
    available = list(CONFIG["models"].keys())

    if model_name not in available:
        raise HTTPException(
            status_code=400,
            detail=f"Model '{model_name}' not found. Available models: {available}"
        )

    try:
        previous_model = manager.current_model_name
        manager.switch_model(model_name)
        return {
            "status":         "success",
            "message":        f"Successfully switched to {model_name}!",
            "previous_model": previous_model,
            "current_model":  manager.current_model_name,
            "val_accuracy":   f"{manager.model_info.get('val_accuracy', 0)}%",
            "timestamp":      datetime.now().isoformat(),
        }
    except FileNotFoundError as e:
        raise HTTPException(
            status_code=404,
            detail=f"Model files not found: {str(e)}. Make sure you have trained this model first."
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# 5. UPLOAD
@app.post("/upload", tags=["Document"])
async def upload_document(file: UploadFile = File(...)):
    """Upload a document image and get file info."""
    if file.content_type not in CONFIG["allowed_types"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type. Allowed: JPG, PNG, BMP, WEBP"
        )
    contents  = await file.read()
    file_size = len(contents) / 1024
    return {
        "status":        "success",
        "message":       "Document uploaded successfully!",
        "filename":      file.filename,
        "file_type":     file.content_type,
        "file_size":     f"{file_size:.2f} KB",
        "timestamp":     datetime.now().isoformat(),
        "active_model":  manager.current_model_name,
        "next_step":     "Send this file to POST /predict to classify it",
    }


# 6. PREDICT
@app.post("/predict", tags=["Document"])
async def predict_document(file: UploadFile = File(...)):
    """
    Classify a document image using the currently active model.
    Switch models first using POST /switch-model/{model_name}
    """
    if file.content_type not in CONFIG["allowed_types"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type. Allowed: JPG, PNG, BMP, WEBP"
        )
    try:
        contents  = await file.read()
        img_array = preprocess_image(contents)
        predicted_class, confidence, all_probs = manager.predict(img_array)

        return {
            "status":            "success",
            "filename":          file.filename,
            "model_used":        manager.current_model_name,
            "predicted_class":   predicted_class,
            "confidence":        round(confidence, 2),
            "confidence_level":  get_confidence_level(confidence),
            "all_probabilities": all_probs,
            "timestamp":         datetime.now().isoformat(),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")


# 7. COMPARE ALL MODELS ← New!
@app.post("/compare", tags=["Document"])
async def compare_models(file: UploadFile = File(...)):
    """
    Upload one image and get predictions from ALL models simultaneously.
    Great for comparing model performance!
    """
    if file.content_type not in CONFIG["allowed_types"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type. Allowed: JPG, PNG, BMP, WEBP"
        )
    try:
        contents  = await file.read()
        img_array = preprocess_image(contents)

        results = {}
        original_model = manager.current_model_name

        for model_name in CONFIG["models"].keys():
            try:
                manager.switch_model(model_name)
                predicted_class, confidence, all_probs = manager.predict(img_array)
                results[model_name] = {
                    "predicted_class":   predicted_class,
                    "confidence":        round(confidence, 2),
                    "confidence_level":  get_confidence_level(confidence),
                    "all_probabilities": all_probs,
                }
            except FileNotFoundError:
                results[model_name] = {
                    "error": "Model not trained yet. Please train this model first."
                }

        # Switch back to original model
        manager.switch_model(original_model)

        # Find best prediction
        best_model = max(
            {k: v for k, v in results.items() if "error" not in v},
            key=lambda k: results[k]["confidence"],
            default=original_model
        )

        return {
            "status":       "success",
            "filename":     file.filename,
            "timestamp":    datetime.now().isoformat(),
            "results":      results,
            "best_model":   best_model,
            "best_confidence": results[best_model].get("confidence", 0),
            "recommendation": f"Use {best_model} — highest confidence for this image",
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Comparison failed: {str(e)}")


# ─────────────────────────────────────────────
# RUN SERVER
# ─────────────────────────────────────────────

print("\n" + "═" * 50)
print("  🚀 Starting FastAPI Server v2.0...")
print("  📌 URL        : http://127.0.0.1:8001")
print("  📄 Swagger UI : http://127.0.0.1:8001/docs")
print("  📘 Redoc      : http://127.0.0.1:8001/redoc")
print("═" * 50 + "\n")

config = uvicorn.Config(
    app,
    host=CONFIG["host"],
    port=CONFIG["port"],
    log_level="info"
)
server = uvicorn.Server(config)
await server.serve()

══════════════════════════════════════════════════
  AI Document Classifier — FastAPI Backend
  With Model Switching Support
══════════════════════════════════════════════════

Loading MobileNetV2...
✔ MobileNetV2 loaded! Accuracy: 97.33%

══════════════════════════════════════════════════
  🚀 Starting FastAPI Server v2.0...
  📌 URL        : http://127.0.0.1:8001
  📄 Swagger UI : http://127.0.0.1:8001/docs
  📘 Redoc      : http://127.0.0.1:8001/redoc
══════════════════════════════════════════════════



INFO:     Started server process [81452]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:58406 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:58406 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:58407 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:58417 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58422 - "GET /model-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:58469 - "POST /predict HTTP/1.1" 200 OK
Loading ResNet50...
✔ ResNet50 loaded! Accuracy: 72.33%
INFO:     127.0.0.1:58492 - "POST /switch-model/ResNet50 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58499 - "POST /predict HTTP/1.1" 200 OK
Loading VGG16...
✔ VGG16 loaded! Accuracy: 100.0%
INFO:     127.0.0.1:58515 - "POST /switch-model/VGG16 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58520 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:58524 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:58544 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:58544 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [81452]
